# Example of Linear Regression

## Import libraries and download dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px



Here we will try to predict disease progression one year after baseline, for diabetic patients.

In [ ]:
# Load the diabates dataset
from sklearn import datasets
X, y = datasets.load_diabetes(return_X_y=True, as_frame=True, scaled=False)

*Brief description:* Ten baseline variables, age, sex, body mass index, average blood pressure, and six blood serum measurements were obtained for each of n = 442 diabetes patients, as well as the response of interest, a quantitative measure of disease progression one year after baseline.

For more info, see: https://scikit-learn.org/stable/datasets/toy_dataset.html#:~:text=7.1.2.-,Diabetes%20dataset,progression%20one%20year%20after%20baseline.


## Let's briefly explore the dataset

- How many samples do we have?
- How many input features?
- Are there any categorical features?
- Are there NaN values?
- What is the range of the numerical features?
- Are the input features correlated between them?
- Are they correlated with the target variable?

In [ ]:
# Check for any null values in the DataFrame



In [ ]:
X.columns

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(15, 20))
numerical_cols = X.columns

subx = 0
suby = 0
for i in numerical_cols:
  #create boxplot in each subplot
  sns.histplot(data=X, x=i, ax=axes[subx,suby])
  if suby == 0:
    suby = 1
  else:
    suby = 0
    subx = subx + 1

The second feature is boolean (it takes only two values), the histogram of the 8th feature needs more exploration, and the rest seem to have few outliers.

## Let's prepare the data for Machine Learning

- Perform a 80/20 train-test split
- Normalize the input features if needed (minmax scaler or standarscaler)
- When normalizing input features, remember to do the `.fit_transform` on the train set BUT ONLY `.transform` on the test set!!

In [ ]:
# Split the data


Given the information extracted from the previous section, we can transform all features with standarscaler, except 'sex'(boolean), and 's4'(minmax scaler seems more reasonable)


In [ ]:
# Sex column


In [ ]:
X_train

In [ ]:
from sklearn.preprocessing import MinMaxScaler


Note: The `reshape(-1, 1)` function is used to convert the shape of the array that the `fit_transform` method operates on.

The `fit_transform` method of the `MinMaxScaler` expects its input to be a 2D array. However, when you select a single column from a pandas DataFrame (like `X_train['s4']`), it returns a 1D array. Thus, you need to reshape this 1D array into a 2D array before you can pass it to the `fit_transform` method.

`.reshape(-1, 1)` does exactly this. The -1 in reshape function is a placeholder for "whatever dimension is needed" and 1 is the size of the second dimension.

So, reshape(-1, 1) changes the shape of the array to have as many rows as needed (which will be the same number of rows as the original array) and 1 column. This way, you get a 2D array with a single column, which is what `fit_transform` expects.

In [ ]:
X_train.columns

In [ ]:
from sklearn.preprocessing import StandardScaler

X_num = X_train[['age', 'bmi', 'bp', 's1', 's2', 's3', 's5', 's6']]


In [ ]:
# Let's put everything back into a dataframe
df_s4_scaled = pd.DataFrame(s4_scaled, index=X_train.index, columns=['s4'])
df_X_num_scaled = pd.DataFrame(X_num_scaled, index=X_num.index, columns=X_num.columns)
# We also have X_train['sex']

## Let's fit some Linear models to the training set

Fit a Linear Regression model without regularization

In [ ]:
from sklearn.linear_model import LinearRegression

What should the dimension of parameter vector $\theta$ be in this case?

Fit a Linear Regression model with L2 regularization (Ridge), with regularization parameter = 1.2

Compare the MAE of both models on the train set

In [ ]:
# MAE
from sklearn.metrics import mean_absolute_error




## Evaluate the models on the test set

In [ ]:
# Compute the prediction error (MAE) of both models on the test set
# Remember you may need to make some transformations on the test set before making predictions

# Sex column
X_test.replace({'sex': {1.0: 1.0, 2.0: 0.0}}, inplace=True)

# s4 column

# remaining num columns
X_num = X_test[['age', 'bmi', 'bp', 's1', 's2', 's3', 's5', 's6']]

# Back into dataframe
df_s4_scaled_test = pd.DataFrame(s4_scaled_test, index=X_test.index, columns=['s4'])
df_X_num_scaled_test = pd.DataFrame(X_num_scaled_test, index=X_num.index, columns=X_num.columns)
df_test = pd.concat([df_s4_scaled_test, df_X_num_scaled_test, X_test['sex']], axis=1)

# Finding the best value of regularization parameter $\alpha$ via CV

**1) Using GridSearch CV**

In [ ]:
# Import libraries
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import KFold

In [ ]:
# Define the hyperparameters
alpha_values = np.logspace(-4, 4, 100)
param_grid = {'alpha': alpha_values}

In [ ]:
# Initialize a Ridge regressor
ridge = Ridge()

# Initialize a 3-fold CV grid search
grid = GridSearchCV(ridge, param_grid, cv=KFold(n_splits=3))

# Fit it with the data
grid.fit(df_train, y_train)

# Print the best hyperparameters
print('Best parameters found by grid search:', grid.best_params_)

In [ ]:
from pandas import DataFrame

# Create a DataFrame from cv_results_
cv_results = DataFrame(grid.cv_results_)

In [ ]:
cv_results.head()

In [ ]:
cv_results.loc[cv_results['rank_test_score']==1]

In [ ]:
# Print the table with alpha, mean_test_score (MSE), and std_test_score
tmp = cv_results[['param_alpha', 'mean_test_score', 'std_test_score', 'rank_test_score']]
tmp.head()

**2) Using Randomized Search CV**

In [ ]:
# Define the probability to draw alpha values from
from scipy.stats import uniform

# Define the hyperparameters
param_dist = {'alpha': uniform(loc=0, scale=4)}  # alpha will range from loc to loc + scale uniformly

from scipy.stats import loguniform

# Define the hyperparameters
param_dist = {'alpha': loguniform(1e-4, 1e4)} # generates a distribution in a logarithmic scale between 1e-4 and 1e4

In [ ]:
# Initialize a Ridge regressor
ridge = Ridge()

# Initialize a 3-fold CV randomized search


In [ ]:
random_search_results = DataFrame(random_search.cv_results_)

In [ ]:
random_search_results.loc[random_search_results['rank_test_score']==1]

RETRAIN the model with the best alpha